# EMNLP 2025 Research Gap Analysis — Cluster 7

This notebook focuses on Cluster 7, a small coherent cluster related to LLM-as-a-Judge / automatic evaluation.

Goal:

1. Extract all Cluster 7 papers
2. Review their titles and abstracts
3. Build a structured comparison table
4. Identify repeated limitations or underexplored dimensions
5. Generate evidence-based candidate research gaps for group discussion

Expected input:
`../data/processed/emnlp_2025_clustered.csv`


## 1. Load the clustered corpus


In [13]:
import pandas as pd
from pathlib import Path

CLUSTERED_FILE = Path("../data/processed/emnlp_2025_clustered.csv")

df = pd.read_csv(CLUSTERED_FILE)

print("Loaded corpus:", df.shape)
print("Available clusters:", sorted(df["cluster"].unique()))


Loaded corpus: (1809, 11)
Available clusters: [np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18)]


## 2. Extract Cluster 7


In [14]:
TARGET_CLUSTER = 7

cluster_df = (
    df[df["cluster"] == TARGET_CLUSTER]
    .copy()
    .reset_index(drop=True)
)

print("Target cluster:", TARGET_CLUSTER)
print("Number of papers:", len(cluster_df))

cluster_df[["paper_id", "title", "abstract"]]


Target cluster: 7
Number of papers: 18


,paper_id,title,abstract
0,2025.emnlp-main.86,Beyond the Surface: Measuring Self-Preference ...,Recent studies show that large language models...
1,2025.emnlp-main.103,Direct Judgement Preference Optimization,To meet the increasing need for timely and acc...
2,2025.emnlp-main.138,From Generation to Judgment: Opportunities and...,Assessment and evaluation have long been criti...
3,2025.emnlp-main.360,Arena-lite: Efficient and Reliable Large Langu...,As Large Language Models (LLMs) expand across ...
4,2025.emnlp-main.538,Are Checklists Really Useful for Automatic Eva...,Automatic evaluation of generative tasks using...
5,2025.emnlp-main.569,Analyzing Uncertainty of LLM -as-a-Judge: Inte...,LLM-as-a-judge has become a promising paradigm...
6,2025.emnlp-main.583,Meta-Rewarding Language Models: Self-Improving...,Large Language Models (LLMs) are rapidly surpa...
7,2025.emnlp-main.625,Self-Augmented Preference Alignment for Sycoph...,Sycophancy causes models to produce answers th...
8,2025.emnlp-main.712,Improve LLM -as-a-Judge Ability as a General A...,LLM-as-a-Judge leverages the generative and re...
9,2025.emnlp-main.716,Towards Optimal Evaluation Efficiency for Larg...,Comprehensive evaluation of large language mod...


## 3. Export the raw Cluster 7 papers

This creates a compact CSV containing only the 18 papers in Cluster 7.


In [15]:
CLUSTER_RAW_FILE = Path(
    "../data/processed/cluster7_llm_judge_papers.csv"
)

cluster_df[
    ["paper_id", "title", "abstract", "paper_url", "pdf_url"]
].to_csv(CLUSTER_RAW_FILE, index=False)

print("Saved:", CLUSTER_RAW_FILE)


Saved: ../data/processed/cluster7_llm_judge_papers.csv


## 4. Create a structured review table

The fields below are designed specifically for research-gap discovery.

Fill them after reading each abstract and, where necessary, the full paper.

Recommended interpretation:

- `main_problem`: What problem does the paper study?
- `judge_model`: Which model(s) act as evaluator/judge?
- `task_or_domain`: What task/domain is being evaluated?
- `evaluation_method`: How judge quality is assessed
- `human_comparison`: Whether judge outputs are compared with human judgments
- `bias_tested`: Whether systematic evaluator bias is studied
- `robustness_tested`: Whether robustness/consistency is tested
- `multilingual`: Whether multiple languages are studied
- `limitations`: Important limitations reported by the authors
- `future_work`: Explicit future research directions


In [16]:
review_columns = [
    "paper_id",
    "title",
    "main_problem",
    "judge_model",
    "task_or_domain",
    "evaluation_method",
    "human_comparison",
    "bias_tested",
    "robustness_tested",
    "multilingual",
    "limitations",
    "future_work",
    "notes"
]

review_df = cluster_df[["paper_id", "title"]].copy()

for col in review_columns[2:]:
    review_df[col] = ""

review_df


,paper_id,title,main_problem,judge_model,task_or_domain,evaluation_method,human_comparison,bias_tested,robustness_tested,multilingual,limitations,future_work,notes
0,2025.emnlp-main.86,Beyond the Surface: Measuring Self-Preference ...,,,,,,,,,,,
1,2025.emnlp-main.103,Direct Judgement Preference Optimization,,,,,,,,,,,
2,2025.emnlp-main.138,From Generation to Judgment: Opportunities and...,,,,,,,,,,,
3,2025.emnlp-main.360,Arena-lite: Efficient and Reliable Large Langu...,,,,,,,,,,,
4,2025.emnlp-main.538,Are Checklists Really Useful for Automatic Eva...,,,,,,,,,,,
5,2025.emnlp-main.569,Analyzing Uncertainty of LLM -as-a-Judge: Inte...,,,,,,,,,,,
6,2025.emnlp-main.583,Meta-Rewarding Language Models: Self-Improving...,,,,,,,,,,,
7,2025.emnlp-main.625,Self-Augmented Preference Alignment for Sycoph...,,,,,,,,,,,
8,2025.emnlp-main.712,Improve LLM -as-a-Judge Ability as a General A...,,,,,,,,,,,
9,2025.emnlp-main.716,Towards Optimal Evaluation Efficiency for Larg...,,,,,,,,,,,


## 5. LLM-assisted first-pass extraction from abstracts

This section uses the OpenAI API to populate the review table from each paper's **title and abstract only**.

The extraction is intentionally conservative:

- use `Yes`, `No`, `Partial`, or `Unclear` for categorical fields
- do not infer information that is not supported by the abstract
- use `Not stated in abstract` for limitations or future work that are not explicitly mentioned
- manually verify the extracted values before using them as evidence for a research gap

The API key is expected in your existing `.env` file as `OPENAI_API_KEY`.


In [17]:
import os
import json
import time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found. Check your .env file.")

client = OpenAI(api_key=api_key)

# Use the same model as the existing Track 3 generation workflow.
MODEL = "gpt-5.6-luna"

print("OpenAI client ready.")


OpenAI client ready.


In [22]:
EXTRACTION_FIELDS = [
    "main_problem",
    "judge_model",
    "task_or_domain",
    "evaluation_method",
    "human_comparison",
    "bias_tested",
    "robustness_tested",
    "multilingual",
    "limitations",
    "future_work",
    "notes"
]

def extract_gap_fields(title, abstract):
    prompt = f"""
You are analysing one research paper for a literature-gap study on LLM-as-a-Judge / automatic evaluation.

Use ONLY the title and abstract below. Do not use outside knowledge and do not infer unsupported details.

TITLE:
{title}

ABSTRACT:
{abstract}

Return ONLY valid JSON with exactly these keys:
{json.dumps(EXTRACTION_FIELDS)}

Rules:
- main_problem: concise description of the research problem.
- judge_model: judge/evaluator model(s) explicitly named; otherwise "Unclear".
- task_or_domain: task or application domain explicitly studied.
- evaluation_method: how judge/evaluator quality is assessed.
- human_comparison: one of Yes / No / Partial / Unclear.
- bias_tested: one of Yes / No / Partial / Unclear.
- robustness_tested: one of Yes / No / Partial / Unclear.
- multilingual: one of Yes / No / Partial / Unclear.
- limitations: explicit limitation(s) stated or directly evidenced in the abstract; otherwise "Not stated in abstract".
- future_work: explicit future-work direction(s); otherwise "Not stated in abstract".
- notes: brief clarification, especially where evidence is uncertain.

Do not treat absence of a term as proof that the full paper does not study it. If the abstract is insufficient, use "Unclear".
"""

    response = client.responses.create(
        model=MODEL,
        input=prompt
    )

    text = response.output_text.strip()

    # Defensive cleanup in case the model wraps JSON in a fenced block.
    if text.startswith("```"):
        text = text.strip("`")
        if text.lower().startswith("json"):
            text = text[4:].strip()

    return json.loads(text)


### Test the extraction on one paper first

Run this before processing all 18 papers. Check that the output is sensible and conservative.


In [23]:
test_paper = cluster_df.iloc[0]

test_result = extract_gap_fields(
    test_paper["title"],
    test_paper["abstract"]
)

print("Title:", test_paper["title"])
print(json.dumps(test_result, indent=2, ensure_ascii=False))


Title: Beyond the Surface: Measuring Self-Preference in LLM Judgments
{
  "main_problem": "Measuring self-preference bias in LLM judges without confounding it with differences in response quality.",
  "judge_model": "Unclear",
  "task_or_domain": "LLM-as-a-judge evaluation of response quality.",
  "evaluation_method": "The DBG score compares the scores a judge assigns to its own responses with corresponding gold judgments intended to represent true response quality; experiments assess bias across LLMs with varying versions, sizes, and reasoning abilities.",
  "human_comparison": "Unclear",
  "bias_tested": "Yes",
  "robustness_tested": "Yes",
  "multilingual": "Unclear",
  "limitations": "Not stated in abstract",
  "future_work": "Not stated in abstract",
  "notes": "The abstract does not name specific judge models, identify the source of the gold judgments, or state whether human judgments or multilingual data are used."
}


### Extract all 18 papers

This loops through Cluster 7 and saves progress after every paper, so a partial run is not lost if the API call stops.


In [24]:
AUTO_REVIEW_FILE = Path(
    "../data/processed/cluster7_gap_review_auto.csv"
)

# Resume from an existing file if present; otherwise start from the blank review table.
if AUTO_REVIEW_FILE.exists():
    auto_review_df = pd.read_csv(AUTO_REVIEW_FILE).fillna("")
    print("Resuming from:", AUTO_REVIEW_FILE)
else:
    auto_review_df = review_df.copy()

for i, paper in cluster_df.iterrows():
    paper_id = paper["paper_id"]

    existing = auto_review_df.loc[
        auto_review_df["paper_id"] == paper_id,
        "main_problem"
    ]

    # Skip already-processed rows when resuming.
    if len(existing) and str(existing.iloc[0]).strip():
        print(f"Skipping {i+1}/{len(cluster_df)}: already processed")
        continue

    print(f"Processing {i+1}/{len(cluster_df)}: {paper['title']}")

    try:
        result = extract_gap_fields(
            paper["title"],
            paper["abstract"]
        )

        row_mask = auto_review_df["paper_id"] == paper_id
        for field in EXTRACTION_FIELDS:
            auto_review_df.loc[row_mask, field] = result.get(field, "Unclear")

        auto_review_df.to_csv(AUTO_REVIEW_FILE, index=False)
        time.sleep(0.5)

    except Exception as e:
        print("  Error:", e)
        auto_review_df.to_csv(AUTO_REVIEW_FILE, index=False)

print("\nSaved automated first-pass review to:")
print(AUTO_REVIEW_FILE)


Processing 1/18: Beyond the Surface: Measuring Self-Preference in LLM Judgments
Processing 2/18: Direct Judgement Preference Optimization
Processing 3/18: From Generation to Judgment: Opportunities and Challenges of LLM -as-a-judge
Processing 4/18: Arena-lite: Efficient and Reliable Large Language Model Evaluation via Tournament-Based Direct Comparisons
Processing 5/18: Are Checklists Really Useful for Automatic Evaluation of Generative Tasks?
Processing 6/18: Analyzing Uncertainty of LLM -as-a-Judge: Interval Evaluations with Conformal Prediction
Processing 7/18: Meta-Rewarding Language Models: Self-Improving Alignment with LLM -as-a-Meta-Judge
Processing 8/18: Self-Augmented Preference Alignment for Sycophancy Reduction in LLM s
Processing 9/18: Improve LLM -as-a-Judge Ability as a General Ability
Processing 10/18: Towards Optimal Evaluation Efficiency for Large Language Models
Processing 11/18: C heck E val: A reliable LLM -as-a-Judge framework for evaluating text generation using c

## 6. Review the automatically extracted table

Treat this as a **first-pass coding**, not final evidence. Verify uncertain claims against the abstract and, for shortlisted gaps, against the full paper.


In [25]:
auto_review_df = pd.read_csv(AUTO_REVIEW_FILE).fillna("")

pd.set_option("display.max_colwidth", 120)

auto_review_df[
    [
        "paper_id",
        "title",
        "main_problem",
        "human_comparison",
        "bias_tested",
        "robustness_tested",
        "multilingual"
    ]
]


,paper_id,title,main_problem,human_comparison,bias_tested,robustness_tested,multilingual
0,2025.emnlp-main.86,Beyond the Surface: Measuring Self-Preference in LLM Judgments,Existing measures of self-preference in LLM judges conflate bias with differences in response quality.,Unclear,Yes,Partial,Unclear
1,2025.emnlp-main.103,Direct Judgement Preference Optimization,"Existing LLM-as-judges are trained with supervised finetuning on small datasets and limited evaluation tasks, which ...",Unclear,Unclear,Unclear,Unclear
2,2025.emnlp-main.138,From Generation to Judgment: Opportunities and Challenges of LLM -as-a-judge,"To survey and organize the use of large language models as judges for scoring, ranking, and selection in machine lea...",Unclear,Unclear,Unclear,Unclear
3,2025.emnlp-main.360,Arena-lite: Efficient and Reliable Large Language Model Evaluation via Tournament-Based Direct Comparisons,Baseline-mediated LLM evaluation can be less reliable than direct system-to-system comparison and may require many c...,Unclear,No,Partial,Unclear
4,2025.emnlp-main.538,Are Checklists Really Useful for Automatic Evaluation of Generative Tasks?,"Determining whether automatically generated checklists improve LLM-based automatic evaluation of generative tasks, a...",Yes,No,Partial,Unclear
5,2025.emnlp-main.569,Analyzing Uncertainty of LLM -as-a-Judge: Interval Evaluations with Conformal Prediction,"The uncertainty and reliability of LLM-based evaluation scores are underexplored, limiting the deployment of LLM-as-...",Unclear,Partial,Unclear,Unclear
6,2025.emnlp-main.583,Meta-Rewarding Language Models: Self-Improving Alignment with LLM -as-a-Meta-Judge,"Existing self-rewarding methods improve model responses but not judgment capabilities, leading to rapid saturation d...",Unclear,Unclear,Unclear,Unclear
7,2025.emnlp-main.625,Self-Augmented Preference Alignment for Sycophancy Reduction in LLM s,"Reducing sycophantic behavior in LLMs, particularly the tendency to favor user expectations over truthful responses,...",Partial,Yes,Unclear,Unclear
8,2025.emnlp-main.712,Improve LLM -as-a-Judge Ability as a General Ability,Developing an accurate and data-efficient LLM-as-a-Judge model with judging ability treated as a general capability.,Unclear,Unclear,Unclear,Unclear
9,2025.emnlp-main.716,Towards Optimal Evaluation Efficiency for Large Language Models,Reducing the data-annotation and computational costs of LLM evaluation while preserving evaluation reliability and b...,Unclear,Unclear,Unclear,Unclear


## 7. Summarise coverage from the automated first pass

This gives a quick view of repeatedly covered or underreported dimensions. `Unclear` is important: it means the abstract does not provide enough evidence.


In [26]:
binary_like_cols = [
    "human_comparison",
    "bias_tested",
    "robustness_tested",
    "multilingual"
]

summary_rows = []

for col in binary_like_cols:
    counts = auto_review_df[col].replace("", "Missing").value_counts()
    print(f"\n=== {col} ===")
    print(counts)

    summary_rows.append({
        "dimension": col,
        "Yes": int(counts.get("Yes", 0)),
        "No": int(counts.get("No", 0)),
        "Partial": int(counts.get("Partial", 0)),
        "Unclear": int(counts.get("Unclear", 0)),
        "Missing": int(counts.get("Missing", 0))
    })

coverage_summary = pd.DataFrame(summary_rows)
coverage_summary



=== human_comparison ===
human_comparison
Unclear    11
Yes         6
Partial     1
Name: count, dtype: int64

=== bias_tested ===
bias_tested
Unclear    12
Yes         3
No          2
Partial     1
Name: count, dtype: int64

=== robustness_tested ===
robustness_tested
Unclear    13
Partial     5
Name: count, dtype: int64

=== multilingual ===
multilingual
Unclear    18
Name: count, dtype: int64


,dimension,Yes,No,Partial,Unclear,Missing
0,human_comparison,6,0,1,11,0
1,bias_tested,3,2,1,12,0
2,robustness_tested,0,0,5,13,0
3,multilingual,0,0,0,18,0


## 8. Research-gap evidence table

Use the automated coding to shortlist possible gaps, but do **not** claim a gap solely because a column contains many `No` or `Unclear` values.

For each promising direction, check the relevant full papers and then validate it against literature beyond this EMNLP 2025 cluster.


In [27]:
gap_evidence = auto_review_df[
    [
        "paper_id",
        "title",
        "main_problem",
        "task_or_domain",
        "bias_tested",
        "robustness_tested",
        "multilingual",
        "limitations",
        "future_work"
    ]
].copy()

gap_evidence


,paper_id,title,main_problem,task_or_domain,bias_tested,robustness_tested,multilingual,limitations,future_work
0,2025.emnlp-main.86,Beyond the Surface: Measuring Self-Preference in LLM Judgments,Existing measures of self-preference in LLM judges conflate bias with differences in response quality.,LLM-as-a-Judge evaluation of model-generated responses,Yes,Partial,Unclear,Not stated in abstract,Not stated in abstract
1,2025.emnlp-main.103,Direct Judgement Preference Optimization,"Existing LLM-as-judges are trained with supervised finetuning on small datasets and limited evaluation tasks, which ...","Evaluating and critiquing large language model responses through pairwise judgement, single-response rating, and cla...",Unclear,Unclear,Unclear,The abstract identifies limited data scale and task coverage as limitations of existing judge models; limitations of...,Not stated in abstract
2,2025.emnlp-main.138,From Generation to Judgment: Opportunities and Challenges of LLM -as-a-judge,"To survey and organize the use of large language models as judges for scoring, ranking, and selection in machine lea...","LLM-based judgment and assessment across various machine learning evaluation scenarios, especially open-ended and dy...",Unclear,Unclear,Unclear,Not stated in abstract,"Highlights promising future directions for the emerging LLM-as-a-judge area, without specifying them in the abstract."
3,2025.emnlp-main.360,Arena-lite: Efficient and Reliable Large Language Model Evaluation via Tournament-Based Direct Comparisons,Baseline-mediated LLM evaluation can be less reliable than direct system-to-system comparison and may require many c...,LLM-based system evaluation and model selection,No,Partial,Unclear,Not stated in abstract,Not stated in abstract
4,2025.emnlp-main.538,Are Checklists Really Useful for Automatic Evaluation of Generative Tasks?,"Determining whether automatically generated checklists improve LLM-based automatic evaluation of generative tasks, a...",Automatic evaluation of generative tasks using pairwise comparison and direct scoring.,No,Partial,Unclear,"The benefits of checklists are less consistent for direct scoring, and some checklist items show low correlation wit...",More clearly define objective evaluation criteria to guide both human and automatic evaluations.
5,2025.emnlp-main.569,Analyzing Uncertainty of LLM -as-a-Judge: Interval Evaluations with Conformal Prediction,"The uncertainty and reliability of LLM-based evaluation scores are underexplored, limiting the deployment of LLM-as-...",Natural language generation evaluation with discrete rating tasks.,Partial,Unclear,Unclear,Not stated in abstract,Not stated in abstract
6,2025.emnlp-main.583,Meta-Rewarding Language Models: Self-Improving Alignment with LLM -as-a-Meta-Judge,"Existing self-rewarding methods improve model responses but not judgment capabilities, leading to rapid saturation d...","LLM alignment, instruction following, and self-evaluation of model judgments.",Unclear,Unclear,Unclear,Existing self-rewarding approaches are described as rapidly saturating during iterative training; no further limitat...,Not stated in abstract
7,2025.emnlp-main.625,Self-Augmented Preference Alignment for Sycophancy Reduction in LLM s,"Reducing sycophantic behavior in LLMs, particularly the tendency to favor user expectations over truthful responses,...",Sycophancy assessment and reduction in open-source language models across multiple tasks.,Yes,Unclear,Unclear,Not stated in abstract,Not stated in abstract
8,2025.emnlp-main.712,Improve LLM -as-a-Judge Ability as a General Ability,Developing an accurate and data-efficient LLM-as-a-Judge model with judging ability treated as a general capability.,Evaluation of LLM responses and provision of reward signals for RLHF across diverse scenarios.,Unclear,Unclear,Unclear,Not stated in abstract,The authors state that they will open-source the model weights and training data.
9,2025.emnlp-main.716,Towards Optimal Evaluation Efficiency f

## 9. Save the editable review table

Open this CSV in Excel, Numbers, or VS Code while reviewing the 18 papers.


In [28]:
REVIEW_FILE = Path(
    "../data/processed/cluster7_gap_review.csv"
)

review_df.to_csv(REVIEW_FILE, index=False)

print("Saved editable review table to:")
print(REVIEW_FILE)


Saved editable review table to:
../data/processed/cluster7_gap_review.csv
